# statcheck-ml quickstart

This notebook shows the whole pipeline on real text.

The package has two halves, and the difference matters:

| Step | Method |
|---|---|
| Find a reported result | learned, or the pattern baseline |
| Check the p-value | exact mathematics, never learned |

No model output ever reaches a verdict. A confident wrong verdict is worse
than no tool at all.

In [ ]:
# From the repository root:
#     pip install -e statcheck-ml
import statcheck_ml

print(statcheck_ml.__version__)

## 1. Recompute a p-value

This is the part that is never learned. Give it a test statistic and its
degrees of freedom, and it returns the p-value that follows.

In [ ]:
from statcheck_ml import compute_p

print('t(23) = 2.45      ->', round(compute_p('t', 2.45, 23), 5))
print('F(2, 30) = 8.72   ->', round(compute_p('F', 8.72, 2, 30), 5))
print('chi2(1) = 3.84    ->', round(compute_p('chi2', 3.84, 1), 5))
print('z = 1.96          ->', round(compute_p('z', 1.96), 5))
print('r(188) = .19      ->', round(compute_p('r', 0.19, 188), 5))

## 2. Check a reported result

`check` compares the reported p-value with the computed one. It allows for
the rounding the author applied, so `p = .02` is not called wrong when the
exact value is .0223.

There are three outcomes:

- `consistent`: the two agree
- `inconsistent`: they disagree
- `decision_error`: they disagree **and** the conclusion changes

In [ ]:
from statcheck_ml import Result, check

cases = [
    ('as reported',      Result('t', 2.45, 23, None, '=', 0.022), '.022'),
    ('p far too large',  Result('t', 2.45, 23, None, '=', 0.500), '.500'),
    ('claimed as significant', Result('t', 1.20, 23, None, '=', 0.030), '.030'),
    ('an upper bound',   Result('F', 8.72, 2, 30, '<', 0.01), '.01'),
]

for label, result, text in cases:
    outcome = check(result, reported_p_text=text)
    print(f'{label:24s} {outcome.verdict:15s} computed={outcome.computed_p:.5f}')

## 3. Select the passages worth reading

Only about 1 line in 700 of a corpus holds a statistic. The prefilter removes
the rest, which is what makes the model practical in a browser.

It is tuned for recall alone. Text it drops can never be recovered, so it
asks "could this hold a result", never "does this look like one".

In [ ]:
from statcheck_ml import Prefilter

text = '''We tested the effect of training on recall.
Participants improved after training, t(47) = 3.48, p = .001, and the
effect held for the delayed test, F(1, 46) = 8.72,
p < .01. See Smith, J. A., & Lee, B. (2019). Memory, 27(4), 120-133.
No difference appeared between the two control groups.'''

pf = Prefilter()
for w in pf.windows(text):
    print(f'line {w.line}: {w.text.splitlines()[min(1, len(w.text.splitlines())-1)][:70]}')

Note that the reference entry was removed. A bibliography has the same
density of digits and punctuation as a result, so it must be stripped or it
fills the candidate set with citations.

## 4. The pattern baseline

`extract` reproduces what a regular expression can find. It exists so that
improvement can be measured, and it is **never** used as a fallback for the
model. Mixing the two would make the comparison meaningless.

In [ ]:
from statcheck_ml.extract import extract

for e in extract(text):
    print(f'{e.test_type:5s} {e.raw!r}')

## 5. Where the pattern fails

In 198 of the 3100 documents in this corpus, the conversion from PDF writes a
control character where the operator belongs. Of results with degrees of
freedom in parentheses, 35.5% are affected.

A pattern cannot read them, because the character it needs is absent. The
meaning is not fixed either: the same control character is an equals sign in
one document and a less-than sign in another, because it comes from the font.

In [ ]:
damaged = 'The label advantage was also observed in accuracy, F(1, 17) \x03 6.38, p \x03 .02.'

print('what the text holds :', repr(damaged[-40:]))
print('pattern finds       :', extract(damaged))
print()
print('This is why the model names the operator in its tag set:')
print('POP_EQ, POP_LT and POP_GT, rather than one POP label.')

## 6. Run the trained model

The model labels every character. Spans follow from the labels, so no
separate window classifier is needed: a window holds a result when the model
tags one.

In [ ]:
import torch
from statcheck_ml.model import CharTagger
from statcheck_ml.labels import TAG_TO_ID, ID_TO_TAG, tags_to_spans
from statcheck_ml.data import encode, normalise

ckpt = torch.load('../models/lite/model.pt', weights_only=False)
vocab = ckpt['vocab']
model = CharTagger(len(vocab), len(TAG_TO_ID))
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(model.size_report())

In [ ]:
def tag(passage):
    passage = normalise(passage)
    ids = torch.tensor([encode(passage, vocab)], dtype=torch.long)
    with torch.no_grad():
        best = model(ids).argmax(-1)[0]
    tags = [ID_TO_TAG[int(t)] for t in best[:len(passage)]]
    return [(passage[s:e], label) for s, e, label in tags_to_spans(tags)]

for piece, label in tag(damaged):
    print(f'  {label:8s} {piece!r}')

## 7. The whole pipeline

Find the results with the model, then check them with the mathematics.

In [ ]:
from statcheck_ml.labels import ENTITY_OPERATOR

def read_and_check(passage):
    parts = dict()
    for piece, label in tag(passage):
        parts.setdefault(label, piece)
    if 'STAT' not in parts or 'TEST' not in parts:
        return None
    operator = next((ENTITY_OPERATOR[k] for k in parts if k.startswith('POP_')), None)
    result = Result(
        test_type=parts['TEST'],
        statistic=float(parts['STAT']),
        df1=float(parts['DF1']) if 'DF1' in parts else None,
        df2=float(parts['DF2']) if 'DF2' in parts else None,
        p_operator=operator,
        p_value=float(parts['PVAL']) if 'PVAL' in parts else None,
    )
    return check(result, reported_p_text=parts.get('PVAL'))

for passage in ['Recall improved, t(47) = 3.48, p = .001.',
                'No effect appeared, F(1, 46) = 0.20, p = .002.']:
    outcome = read_and_check(passage)
    print(passage)
    print('   ->', outcome.verdict if outcome else 'nothing found',
          f'(computed {outcome.computed_p:.4f})' if outcome and outcome.computed_p else '')

## What to remember

1. The model finds results. It never judges them.
2. The prefilter sets the recall ceiling, so it is tuned for recall alone.
3. Every label in the dataset came from a language model, not a person. No
   measurement in this project can find a mistake the annotator makes every
   time.
4. A result that is found is not always a result that can be checked. The
   arithmetic needs a statistic, degrees of freedom and a p-value.